# GPT-SoVITS trên Colab (T4)

Repo: https://github.com/RVC-Boss/GPT-SoVITS

## Về mẫu giọng 5 giây (nói thẳng)
Theo README của repo:
- **Zero-shot (không huấn luyện):** cần mẫu giọng khoảng **5 giây**. Đây là mức bạn đang có, **đủ để suy luận zero-shot** (dùng làm audio tham chiếu).
- **Few-shot / fine-tune:** README nêu khoảng **1 phút** dữ liệu huấn luyện. Fine-tune với 5 giây **không phù hợp** (dễ quá khớp, giọng kém ổn định). Muốn fine-tune hãy thu ít nhất ~1 phút giọng sạch, một người nói, không nhạc nền.
- Nếu chỉ có 5 giây: bỏ qua mục huấn luyện, đi thẳng tới mục **Suy luận zero-shot**.

## Chuẩn bị mẫu giọng
- Định dạng wav/mp3, mono, giọng sạch, không vang, không nhạc. Với zero-shot, mẫu tham chiếu nên dài 3-10 giây và bạn phải biết **nội dung chữ** của nó (dùng làm `prompt_text`).
- Đặt vào `Drive/content-factory/input/voice/` (vd. `mau_giong.wav`) kèm `mau_giong.txt` chứa lời nói.

> **Bước 0:** vào `Runtime > Change runtime type > T4 GPU` rồi bấm Save. Phiên Colab miễn phí có giới hạn thời gian; mọi kết quả đều được lưu ra Google Drive để không mất khi phiên bị ngắt.

Cấu trúc thư mục trên Drive (tự tạo ở ô mount):

```
MyDrive/content-factory/
  input/    <- đưa kịch bản .txt, video, audio lên đây
  output/   <- kết quả để tải về Local
  models/   <- trọng số / checkpoint
```

## 1. Cấu hình

In [ ]:
#@title Tham số
VOICE_DIR     = "voice"            #@param {type:"string"}
REF_AUDIO     = "mau_giong.wav"    #@param {type:"string"}
REF_TEXT_FILE = "mau_giong.txt"    #@param {type:"string"}
REF_LANG      = "vi"               #@param {type:"string"}
SCRIPT_FILE   = "kichban.txt"      #@param {type:"string"}
TEXT_LANG     = "vi"               #@param {type:"string"}
EXP_NAME      = "giong_cua_toi"    #@param {type:"string"}
GPTSOVITS_COMMIT = ""              #@param {type:"string"}
# LUU Y: README liet ke ma ngon ngu zh, ja, en, ko, yue. Tieng Viet KHONG co trong danh sach do.
# TODO: kiem tra - GPT-SoVITS co ho tro 'vi' hay khong. Neu khong, dung 'en' (chat luong tieng Viet co the kem).

## 2. Mount Drive và kiểm tra GPU

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/content-factory'
IN_DIR, OUT_DIR, MODEL_DIR = f'{BASE}/input', f'{BASE}/output', f'{BASE}/models'
for d in (IN_DIR, OUT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)
print('Input :', IN_DIR)
print('Output:', OUT_DIR)
print('Models:', MODEL_DIR)

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'KHONG THAY GPU. Hay chon Runtime > Change runtime type > T4 GPU.')

## 3. Cài đặt
README dùng conda (`install.sh`). Trên Colab ta dùng cách **cài thủ công** của README: `pip install -r extra-req.txt --no-deps` rồi `pip install -r requirements.txt`, cùng FFmpeg và libsox-dev. README đã kiểm thử Python 3.12 + PyTorch 2.7.1 (CUDA 12.8).

In [ ]:
import subprocess, os
def sh(cmd, cwd=None, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print('LOI:', r.stderr[-2500:])
        if check:
            raise RuntimeError(f'Lenh that bai: {cmd}')
    return r

sh('python --version')
sh('apt-get -y -qq install ffmpeg libsox-dev')
if not os.path.isdir('/content/GPT-SoVITS'):
    sh('git clone https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS')
if GPTSOVITS_COMMIT:
    sh(f'git checkout {GPTSOVITS_COMMIT}', cwd='/content/GPT-SoVITS')
sh('git rev-parse HEAD', cwd='/content/GPT-SoVITS')
sh('pip install -q -r extra-req.txt --no-deps', cwd='/content/GPT-SoVITS')
sh('pip install -q -r requirements.txt', cwd='/content/GPT-SoVITS')
# TODO: kiem tra - Colab co the can khop phien ban torch/CUDA voi README; neu xung dot, doc loi o tren.

## 4. Tải mô hình tiền huấn luyện
README: tải từ Hugging Face `lj1995/GPT-SoVITS` vào `GPT_SoVITS/pretrained_models` (đây là bước thay cho `install.sh`).

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id='lj1995/GPT-SoVITS', local_dir='/content/GPT-SoVITS/GPT_SoVITS/pretrained_models')
print('Da tai xong mo hinh tien huan luyen.')
# TODO: kiem tra - du dung luong (nhieu GB). Neu can ASR tieng Anh/Nhat (Faster Whisper Large V3), dat vao tools/asr/models.

## 5A. (Chỉ khi có >= ~1 phút giọng) Tiền xử lý
Cắt audio rồi ASR để tạo file `.list` (`vocal_path|speaker_name|language|text`). Lệnh dưới lấy từ README; các tham số số liệu là giá trị gợi ý, hãy xem `python tools/slice_audio.py --help` nếu lỗi.

In [ ]:
import subprocess, os
SLICED = '/content/work/sliced'
ASR_OUT = '/content/work/asr'
os.makedirs(SLICED, exist_ok=True); os.makedirs(ASR_OUT, exist_ok=True)
RAW = f'{IN_DIR}/{VOICE_DIR}'
if not os.path.isdir(RAW):
    print('Khong co thu muc giong:', RAW)
else:
    # README ghi 'python audio_slicer.py' voi cac tham so --input_path --output_root --threshold --min_length --min_interval --hop_size
    # TODO: kiem tra - duong dan that cua audio_slicer.py trong repo (co the la tools/slicer2.py).
    r = subprocess.run(f'python tools/slice_audio.py {RAW} {SLICED}', shell=True, cwd='/content/GPT-SoVITS', text=True, capture_output=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    # ASR ngon ngu khac tieng Trung (README): python ./tools/asr/fasterwhisper_asr.py -i <input> -o <output> -l <language> -p <precision>
    r = subprocess.run(f'python ./tools/asr/fasterwhisper_asr.py -i {SLICED} -o {ASR_OUT} -l en -p float16',
                       shell=True, cwd='/content/GPT-SoVITS', text=True, capture_output=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    # Sau do HAY HIEU DINH tay file .list trong ASR_OUT (README: buoc 5 'proofread ASR').

## 5B. (Chỉ khi có >= ~1 phút giọng) Huấn luyện bằng WebUI
README hướng dẫn fine-tune qua WebUI: `python webui.py <ngôn ngữ>` (cổng mặc định `webui_port_main = 9874` trong `config.py`). Trên Colab, cần đường hầm để mở giao diện.
Các tab: `1-GPT-SoVITS-TTS` -> định dạng dữ liệu -> huấn luyện SoVITS -> huấn luyện GPT. Với T4, để batch size nhỏ.

In [ ]:
# TODO: kiem tra - cach mo WebUI tu Colab. Phuong an: dung cloudflared de tao link cong khai tam thoi.
import subprocess, time
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
webui = subprocess.Popen('python webui.py en', shell=True, cwd='/content/GPT-SoVITS')
time.sleep(25)
tunnel = subprocess.Popen('cloudflared tunnel --url http://localhost:9874', shell=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
import re
for _ in range(60):
    line = tunnel.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
    if m:
        print('MO WEBUI TAI:', m.group(0)); break
print('Huan luyen xong thi chay o 6 de xuat mo hinh. Dung WebUI: webui.terminate()')

## 6. Xuất model (.ckpt / .pth) sang Drive/models
Theo `config.py`: trọng số SoVITS lưu ở `SoVITS_weights*` (`.pth`), trọng số GPT ở `GPT_weights*` (`.ckpt`), tên thư mục phụ thuộc phiên bản (v1: `SoVITS_weights`/`GPT_weights`, v2: `..._v2`, v3: `..._v3`, v4: `..._v4`, v2Pro, v2ProPlus, v5dev, v5turbo).

In [ ]:
import glob, os, shutil
dst = f'{MODEL_DIR}/{EXP_NAME}'
os.makedirs(dst, exist_ok=True)
n = 0
for pat in ('SoVITS_weights*/*.pth', 'GPT_weights*/*.ckpt'):
    for f in glob.glob(f'/content/GPT-SoVITS/{pat}'):
        shutil.copy2(f, dst); n += 1
        print('DA LUU:', os.path.join(dst, os.path.basename(f)))
if n == 0:
    print('Chua co trong so da huan luyen (binh thuong neu ban chi dung zero-shot).')

## 7. Suy luận zero-shot: tạo audio từ kịch bản
Chạy `api_v2.py` (repo có tệp này; cổng mặc định trong `config.py` là `api_port = 9880`) và gọi `/tts`. Tên trường JSON lấy theo hiểu biết về api_v2, **chưa xác minh** trong lượt đọc README -> `# TODO: kiểm tra`.

In [ ]:
import subprocess, time, requests, os, pathlib, wave
api = subprocess.Popen('python api_v2.py -a 127.0.0.1 -p 9880 -c GPT_SoVITS/configs/tts_infer.yaml',
                       shell=True, cwd='/content/GPT-SoVITS')  # TODO: kiem tra - co & tham so cua api_v2.py
time.sleep(40)

ref_audio = f'{IN_DIR}/{VOICE_DIR}/{REF_AUDIO}'
ref_text = pathlib.Path(f'{IN_DIR}/{VOICE_DIR}/{REF_TEXT_FILE}').read_text(encoding='utf-8').strip()
script = pathlib.Path(f'{IN_DIR}/{SCRIPT_FILE}').read_text(encoding='utf-8').strip()
payload = {
    'text': script, 'text_lang': TEXT_LANG,
    'ref_audio_path': ref_audio, 'prompt_text': ref_text, 'prompt_lang': REF_LANG,
    'text_split_method': 'cut5', 'batch_size': 1, 'media_type': 'wav', 'streaming_mode': False,
}  # TODO: kiem tra ten truong voi api_v2.py
try:
    r = requests.post('http://127.0.0.1:9880/tts', json=payload, timeout=1800)
    r.raise_for_status()
    out = f'{OUT_DIR}/{EXP_NAME}_voice.wav'
    open(out, 'wb').write(r.content)
    print('KET QUA:', out, f'({len(r.content)/1e6:.1f} MB)')
except Exception as e:
    print('LOI suy luan:', e)
    print('Neu loi, mo inference_webui.py (python GPT_SoVITS/inference_webui.py, cong 9872) thu cong.')
finally:
    api.terminate()